# Tutorial 1 - SPAC-seq held-out perturbation benchmark

This notebook reproduces the SPAC-seq benchmark numbers from a **trained CONCERT checkpoint**.
Nothing is retrained here: every cell below loads `model.pt` and runs inference.

### The question being scored

Five tissue sections (`1-1` ... `1-5`) carry a pooled CRISPR perturbation panel. A set of
perturbations is held out. For each held-out cell the model is given

* **where the cell sits** in the section (its array coordinates), and
* **what it looked like unperturbed** (its measured expression),

and is asked to predict the perturbed expression. The prediction is compared with the cells
that actually carry that perturbation, through the **log fold change over matched unperturbed
cells**.

### The two settings

| Code name | Manuscript name | What is held out |
|---|---|---|
| `seen` | **within-niche prediction** | the perturbation is removed from a patch of one slide but is still visible elsewhere - does its effect transfer to a new tissue context? |
| `unseen` | **cross-slide prediction** | the perturbation is removed from *every* slide - can its effect be inferred from related perturbations? |

The code paths, output directories and CSV columns still say `seen` / `unseen`. Those names are
kept here because they are what is on disk; the manuscript uses the two names in the middle column.

### The two metrics

* **LFC Spearman** - do the right genes move, in the right direction? (higher is better)
* **LFC RMSE** (`lfc_dist` in the tables) - is the size of the response right? (lower is better)

E-distance appears in some older JSON files. It is **retired**: it is not a reported metric and
should not be quoted.

### What the notebook does

1. resolve every path and check it exists
2. load the trained checkpoint into a live model
3. read the learned per-perturbation spatial footprint lengthscales out of the weights
4. make **one** counterfactual prediction by hand and look at it
5. score **one slide** with the exact function the SLURM array job calls
6. aggregate the five slides into the reported within-niche number
7. repeat for the cross-slide setting
8. put CONCERT next to the published methods under the shared scoring rule

## 0. Before you start

**Run this on a GPU node.** Steps 4-7 do real inference; on CPU they are unusably slow.

```bash
# interactive GPU session, then start Jupyter inside it
salloc -p gpu --gres=gpu:1 --mem=200G -c 8 -t 0-08:00
source /n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/miniconda3_new/etc/profile.d/conda.sh
conda activate /n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/concert_env
jupyter lab --no-browser --ip=0.0.0.0
```

The environment is `concert_env`. Note that it is pinned to CUDA 11.8 / torch 2.3 and **cannot
use Blackwell GPUs** - ask for an H100 or A100 (`--gres=gpu:nvidia_h200:1` on the Kempner
partitions) if the job dies with an `nvrtc -arch` error a few minutes in.

**Rough runtimes on one H100**

| Step | Time |
|---|---|
| 2 - load checkpoint + slide | 2-4 min |
| 4 - one prediction | < 1 min |
| 5 - score one slide (full panel) | 15-40 min |
| 6, 7, 8 - reading shipped results | seconds |

Step 5 is the only slow one, and it is optional: set `RUN_SLIDE_EVAL = False` in that cell to
skip it and read the shipped result instead.

## 1. Paths

Everything the notebook touches is named here. Three roots are involved:

* **`CODEBASE`** - the released code (model source + scoring code).
* **`OUTPUTS`** - where training wrote the checkpoints.
* **`SCORING_LIB`** - the evaluation working tree, which holds the prepared cell sets and the
  results that came with the paper.
* **`HELPER_LIB`** - two metric helpers (`eval_utils`, `spac_edist_align`) that the scoring code
  imports from an absolute path baked into `spacseq_eval_common.EVAL_UTILS_DIR`. The module puts
  that directory on the path itself; it just has to exist.

A checkpoint directory must contain **both** `model.pt` and `pert_map.json`. The perturbation
vocabulary is *not* rebuilt at evaluation time - it is read from `pert_map.json`, so that the
integer code of every perturbation matches the one the weights were trained with.

In [ ]:
import os, sys, json
from pathlib import Path
from types import SimpleNamespace

# --- code ---------------------------------------------------------------------------------
# The repository root: the directory holding models/ and benchmark_evaluation/. Found by
# walking up from the working directory, so the notebook runs from anywhere inside the repo;
# set CONCERT_CODEBASE to override. Every other root below reads an environment variable too,
# with the paths these results were produced on as the fallback -- see datasets/README.md.
def _repo_root():
    env = os.environ.get("CONCERT_CODEBASE")
    if env:
        return Path(env)
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "models" / "concert_benchmark").is_dir():
            return d
    raise RuntimeError("cannot locate the repository root; set CONCERT_CODEBASE")

def _root(var, default):
    return Path(os.environ.get(var, default))

CODEBASE    = _repo_root()
MODEL_SRC   = CODEBASE / "models" / "concert_benchmark"      # the model this checkpoint is
EVAL_SRC    = CODEBASE / "benchmark_evaluation"              # shared scoring code
SCORING_LIB = _root("CONCERT_SCORING_LIB", "/n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/spac_seq_eval")
# Two metric helpers (eval_utils, spac_edist_align) are imported from an absolute path baked
# into spacseq_eval_common.EVAL_UTILS_DIR. It is not on sys.path here -- the module inserts it
# itself -- but it has to exist, so it is checked below.
HELPER_LIB  = _root("CONCERT_HELPER_LIB", "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/spaVAE/CONCERT-main/src/CONCERT-spac")

# --- data ---------------------------------------------------------------------------------
SPAC_RAW = _root("CONCERT_DATA_ROOT", "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/spaVAE/datasets") / "newdata" / "spac_seq"
PREPARED = SCORING_LIB / "multi_1-5" / "prepared"            # matched cell sets used at scoring

# --- trained checkpoints ---------------------------------------------------------------------
OUTPUTS = _root("CONCERT_OUTPUT_ROOT", "/n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/CONCERT/outputs")
CKPT = {
    "seen":   OUTPUTS / "ms_spac1_niche_seen_lsfix"   / "model.pt",
    "unseen": OUTPUTS / "ms_spac1_niche_unseen_lsfix" / "model.pt",
}

# --- results that came with the paper --------------------------------------------------------
SHIPPED = {
    "seen":   SCORING_LIB / "multi_1-5_ho_v3" / "concert" / "seen_lsfix",
    "unseen": SCORING_LIB / "multi_1-5_ho_v3" / "concert" / "unseen_lsfix",
}
PAPER_NUMBERS = _root("CONCERT_PAPER_NUMBERS", "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/CONCERT/paper_package/numbers") / "benchmark_spac_seq"

# --- where THIS notebook writes ---------------------------------------------------------------
WORKDIR = Path(os.environ.get(
    "CONCERT_NB_WORKDIR",
    "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/CONCERT/notebook_runs/01_spac_seq"))
WORKDIR.mkdir(parents=True, exist_ok=True)

SLIDES = ["1-1", "1-2", "1-3", "1-4", "1-5"]

checks = [
    ("model source",        MODEL_SRC / "concert_spac.py"),
    ("scoring code",        EVAL_SRC / "run_concert_v3_benchmark_eval.py"),
    ("metric helper lib",   HELPER_LIB / "spac_edist_align.py"),
    ("raw section 1-1",     SPAC_RAW / "raw_output_1-1.h5ad"),
    ("prepared cell sets",  PREPARED / "slide_1-1.h5ad"),
    ("eval panel",          PREPARED / "benchmark_v3" / "eval_panel.json"),
    ("seen checkpoint",     CKPT["seen"]),
    ("seen vocabulary",     CKPT["seen"].parent / "pert_map.json"),
    ("unseen checkpoint",   CKPT["unseen"]),
    ("unseen vocabulary",   CKPT["unseen"].parent / "pert_map.json"),
    ("shipped seen result", SHIPPED["seen"] / "patch" / "slide_1-1" / "lfc_agg.json"),
    ("paper number table",  PAPER_NUMBERS / "spac_seen_lfc_dist_per_record.csv"),
]
for label, p in checks:
    print(f"{'OK ' if p.exists() else 'MISSING':>8}  {label:<22}  {p}")
print(f"\nnotebook output dir: {WORKDIR}")

## 2. Import the model and scoring code

**The import order matters.** Several of these modules put their own directory at the front of
`sys.path` when they are imported, and older copies of the same file names exist in the working
trees. Importing `eval_spatial_counterfactual` *first*, with the release directories at the front
of the path, binds the released copy into `sys.modules`; every later import then reuses it. The
printed `__file__` values are the check that the release copy is what actually got loaded.

In [ ]:
for p in (str(EVAL_SRC), str(MODEL_SRC)):
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(EVAL_SRC))      # released scoring code wins over the working tree
sys.path.insert(0, str(MODEL_SRC))     # released model source wins over everything

import numpy as np
import torch
import eval_spatial_counterfactual as ecf      # binds the released model copy first
import spacseq_eval_common as sec
import spacseq_multi_common as smc
import run_concert_v3_benchmark_eval as bench  # the entry point the SLURM job runs

eu, sa = sec.import_eval_utils()

print("model source      :", ecf.__file__)
print("scoring entrypoint:", bench.__file__)
print("metric helpers    :", eu.__file__)
print("torch             :", torch.__version__)
print("device            :", "cuda" if torch.cuda.is_available() else "CPU ONLY - too slow, get a GPU node")
if torch.cuda.is_available():
    print("gpu               :", torch.cuda.get_device_name(0))

## 3. The evaluation configuration

`build_concert_ctx` rebuilds the model graph and then loads the weights into it, so **every flag
that changes the shape of the graph has to match the flags training used**. The values below are
copied from `experiments/01_spac_seq_benchmark/train/train_spac_seq.sh`.

Two of them are worth calling out, because getting them wrong fails *silently* rather than loudly:

* **`fpert_ls_min` / `fpert_ls_max` - the footprint lengthscale band.** The per-perturbation
  lengthscale is `ls_min + (ls_max - ls_min) * sigmoid(ls_head(e_p))`. The band is a plain float
  attribute of the field, not a saved weight, so identical trained weights render a **different
  spatial field** if it is left at the default. It has to be passed at evaluation as well as at
  training.
* **`fpert_svgp_inducing` - the inducing-grid size.** This one *is* a saved tensor
  (`fpert_field.Z`), so a mismatch would fail the load. `build_concert_ctx` reads the true count
  out of the checkpoint and self-corrects, and does the same for the niche-FiLM head.

`n_boot` / `boot_frac` control the bootstrap that produces the spread on each score: within each
niche the source cells are resampled at 50%, ten times.

In [ ]:
def spac_args(regime, target_slide=None, outdir=None, n_boot=10, force=True):
    """Evaluation configuration for one SPAC-seq run.

    Mirrors the CLI defaults of run_concert_v3_benchmark_eval.py, with the values that
    experiments/01_spac_seq_benchmark/train/train_spac_seq.sh trained with.
    """
    return SimpleNamespace(
        # --- what to score ---------------------------------------------------------------
        regime=regime,
        target_slide=target_slide,
        prepared_dir=PREPARED,
        benchmark_v3_dir=None,                 # defaults to prepared_dir/benchmark_v3
        model_file=CKPT[regime],
        outdir=Path(outdir) if outdir else (WORKDIR / f"{regime}_eval"),
        force=force,
        # --- scoring protocol ------------------------------------------------------------
        n_hvgs=2000,                           # genes the metrics are computed over
        min_g_cells=20,                        # a record needs this many measured KO cells
        max_src_cells=500,                     # cross-slide: source cells per unit
        max_src_cells_per_niche=500,           # within-niche: source cells per niche
        patch_role=1,                          # 1 = the held-out patch
        n_samples=10,                          # decoder draws averaged per prediction
        batch_size=512,
        n_boot=n_boot, boot_frac=0.5, boot_seed=1234,
        # --- model graph: MUST match training --------------------------------------------
        hvg_max_rank=2000,
        min_cells_per_pert=25,
        use_expr_encoder=True, expr_hidden_dims=[512, 256], use_basal_encoder_only=True,
        use_nngp=False, nngp_lambda=0.0,
        use_paracrine=False, paracrine_l1=0.0,
        use_niche_interaction=False,           # replaced by the FiLM head, auto-detected
        niche_rank=8,
        use_pert_gate=True, gate_hidden=64,
        use_action_correction=True,            # SPAC-seq has non-targeting controls
        inducing_point_steps=6,
        spatial_basal_only=True,
        basal_field="rff", basal_bayes=True,
        # --- perturbation footprint field -------------------------------------------------
        fpert_field="gp_pp",
        fpert_rff_features=128, fpert_rff_bandwidth=0.15,
        fpert_coord_scale=20.0, fpert_hidden=256,
        fpert_svgp_inducing=16,                # self-corrected from the checkpoint
        fpert_svgp_ls_init=0.25,
        fpert_mb_bandwidths="", f_pert_clamp=5.0,
        ls_prior_min=3.0, ls_prior_max=18.0,   # legacy band, in loc units
        fpert_ls_min=0.15, fpert_ls_max=0.90,  # THE band that is used, in normalized units
    )

SLIDE = "1-1"
args = spac_args("seen", target_slide=SLIDE, outdir=WORKDIR / "seen_eval")
print(f"scoring {args.regime} / slide {SLIDE}")
print(f"checkpoint  {args.model_file}")
print(f"band        [{args.fpert_ls_min}, {args.fpert_ls_max}] normalized "
      f"= [{args.fpert_ls_min * args.fpert_coord_scale:.0f}, "
      f"{args.fpert_ls_max * args.fpert_coord_scale:.0f}] section units")

## 4. Load the checkpoint

`build_concert_ctx` does four things:

1. reads all five sections with the shared highly-variable-gene mask,
2. maps every cell's perturbation label to its trained integer code using `pert_map.json`,
3. keeps only the target slide (after the vocabulary is built, so the codes stay valid),
4. constructs the model graph and calls `load_model` on it.

What comes back is a plain dict - the evaluation context. The keys used below are:

| key | meaning |
|---|---|
| `model` | the loaded model, in `eval()` mode |
| `N` | cells on the target slide |
| `Y_expr` | normalized expression fed to the encoder |
| `Y_obs` | raw counts, same rows |
| `gene_names` | the 2000 genes the model predicts |
| `coords` / `coords_raw` | normalized and pixel coordinates |
| `perturb_raw` | each cell's perturbation label |
| `cls` | each cell's class: `real` / `control` / `unassigned` |
| `name2code` | perturbation name -> trained integer code (code 0 is the basal slot) |

This takes a couple of minutes: it reads all five sections.

In [ ]:
%%time
ctx = bench.build_concert_ctx(args, SLIDE)

model = ctx["model"]
print()
print(f"slide {SLIDE}: {ctx['N']:,} cells x {len(ctx['gene_names'])} genes")
print(f"perturbation vocabulary: {ctx['n_batch']} codes (code 0 = basal)")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
import collections
print("cell classes:", dict(collections.Counter(ctx["cls"])))

## 5. What the weights say about each perturbation

Two per-perturbation quantities are read straight out of the checkpoint. Both are produced by a
small head applied to the 256-dimensional perturbation embedding
`e_p = lord_encoder.s_encoder.perturbation.weight[p]`:

* **footprint lengthscale** `ls(p) = ls_min + (ls_max - ls_min) * sigmoid(ls_head(e_p))` -
  how far the perturbation's spatial effect field reaches. A lengthscale that sits **inside** the
  band rather than pinned against an end is what allows it to be described as *learned* rather
  than *bounded*.
* **niche gain** `g_scale(p) = min(softplus(g_scale_head(e_p)), g_cap)` - how strongly the local
  microenvironment is allowed to reshape this perturbation's response.

No inference is needed for this; it is arithmetic on the state dict.

In [ ]:
import pandas as pd

sd = torch.load(str(args.model_file), map_location="cpu", weights_only=False)
if isinstance(sd, dict) and "model_state_dict" in sd:
    sd = sd["model_state_dict"]

emb  = sd["lord_encoder.s_encoder.perturbation.weight"].float()   # (n_perts, 256)
ls_w = sd["fpert_field.ls_head.weight"].float()
ls_b = sd["fpert_field.ls_head.bias"].float()
gs_w = sd["g_scale_head.weight"].float()
gs_b = sd["g_scale_head.bias"].float()

LS_MIN, LS_MAX, G_CAP, COORD_SCALE = args.fpert_ls_min, args.fpert_ls_max, 3.0, args.fpert_coord_scale
ls_norm = (LS_MIN + (LS_MAX - LS_MIN) * torch.sigmoid(emb @ ls_w.T + ls_b)).squeeze(1).numpy()
g_scale = torch.clamp(torch.nn.functional.softplus(emb @ gs_w.T + gs_b), max=G_CAP).squeeze(1).numpy()

code2name = ctx["pert_name"]
tbl = pd.DataFrame({
    "code": np.arange(len(ls_norm)),
    "perturbation": [code2name.get(i, f"code{i}") for i in range(len(ls_norm))],
    "lengthscale_norm": ls_norm,
    "lengthscale_section_units": ls_norm * COORD_SCALE,
    "niche_gain": g_scale,
}).iloc[1:]                       # code 0 is the basal slot, not a perturbation

print(f"band in normalized units: [{LS_MIN}, {LS_MAX}]")
print(f"learned lengthscales    : {tbl.lengthscale_norm.min():.3f} - {tbl.lengthscale_norm.max():.3f}")
print(f"pinned at either end    : {int(((tbl.lengthscale_norm - LS_MIN).abs() < 1e-3).sum() + ((tbl.lengthscale_norm - LS_MAX).abs() < 1e-3).sum())} of {len(tbl)}")
tbl.sort_values("lengthscale_norm").head(10)

## 6. One counterfactual prediction, by hand

Before running the whole protocol, it helps to see a single prediction.

The decoder is

```
log mu  =  f_basal(x, expression)  +  1[has_guide] * mu_action  +  1[is_real_KO] * f_pert(g, x)
```

so a prediction for perturbation `g` and a prediction for the basal slot (`code = 0`) differ by
exactly the perturbation terms - the basal reconstruction cancels. That is why every score below
is computed on the **difference** of two decodes rather than on one decode alone:

```
model LFC  =  log mean(pred(cells -> g))  -  log mean(pred(cells -> basal))
```

The source cells are the held-out patch: cells whose own perturbation was masked out of the
model's view of this slide. `map_prep_to_model_rows` bridges the two indexings - the prepared
cell sets are addressed by barcode, the model by row - by matching pixel coordinates.

This cell is an **illustration**, not the reported protocol: it pools the whole held-out patch
and compares against all measured cells of that perturbation on the slide, where the reported
score is computed per niche. Step 7 runs the real thing.

In [ ]:
%%time
# 1. the held-out patch and which perturbation dominates each niche in it
adata_t   = smc.load_prepared_slide(args.prepared_dir, SLIDE)
role      = smc.load_patch_roles(args.prepared_dir, SLIDE, adata_t)
niche_dom = smc.patch_dominant_perts(adata_t, role, args.min_g_cells, src_role=args.patch_role)
src_idx   = smc.patch_source_indices(adata_t, role, niche_dom, args.max_src_cells_per_niche,
                                     seed=42, src_role=args.patch_role)
kept_idx, model_rows = bench.map_prep_to_model_rows(SLIDE, np.asarray(adata_t.obs_names), src_idx, ctx)
print(f"{len(niche_dom)} evaluable niches, {len(model_rows)} source cells mapped to model rows")

# 2. pick one perturbation and decode the same cells twice
GENE = sorted(set(niche_dom.values()))[0]
code_g = ctx["name2code"][GENE]
print(f"predicting {GENE} (code {code_g}) on {len(model_rows)} source cells")

pred_g     = ecf._predict_swap(ctx, model_rows, code_g, args)   # cells -> GENE
pred_basal = ecf._predict_swap(ctx, model_rows, 0, args)        # same cells -> basal
model_lfc  = eu.pseudobulk_lfc(pred_g, pred_basal)

# 3. the measured comparison: cells that really carry GENE, against unassigned cells
ko  = np.where(ctx["perturb_raw"] == GENE)[0]
ref = np.where(ctx["cls"] == "unassigned")[0]
obs_lfc = eu.pseudobulk_lfc(ctx["Y_obs"][ko], ctx["Y_obs"][ref])
print(f"measured: {len(ko)} {GENE} cells vs {len(ref)} unassigned cells")

scores = eu.score_lfc_agreement(model_lfc, obs_lfc)
print("\nslide-level illustration (NOT the reported per-niche score):")
for k, v in scores.items():
    print(f"  {k:<16} {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.scatter(obs_lfc, model_lfc, s=5, lw=0, alpha=0.35, color="#2166ac", rasterized=True)
lim = np.nanpercentile(np.abs(np.r_[obs_lfc, model_lfc]), 99.5)
ax.plot([-lim, lim], [-lim, lim], lw=1, color="0.6", zorder=0)
ax.axhline(0, lw=0.6, color="0.85", zorder=0); ax.axvline(0, lw=0.6, color="0.85", zorder=0)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel(f"measured log fold change ({GENE})", fontsize=13)
ax.set_ylabel("predicted log fold change", fontsize=13)
ax.tick_params(labelsize=11)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
plt.show()

## 7. Score one slide with the shipped harness

`eval_seen_slide` is exactly what `evaluation/eval_spac_seq_benchmark.sh` runs, one slide per
array task. It

1. finds the niches of the held-out patch and the perturbation dominating each,
2. decodes every source cell to every perturbation in the panel, plus the basal slot,
3. scores each (niche, perturbation) record against the measured cells of that perturbation,
   using matched unperturbed cells as the reference,
4. bootstraps the source cells within each niche for the spread,
5. writes `lfc_scores.json`, `lfc_agg.json` and `per_pert_summary.json`.

**This is the slow step** - it decodes the full panel. Set `RUN_SLIDE_EVAL = False` to skip it
and read the shipped result for the same slide instead.

In [ ]:
%%time
RUN_SLIDE_EVAL = True     # False -> read the shipped result instead of recomputing

if RUN_SLIDE_EVAL:
    bench.eval_seen_slide(args, SLIDE)
    agg_path = args.outdir / "patch" / f"slide_{SLIDE}" / "lfc_agg.json"
else:
    agg_path = SHIPPED["seen"] / "patch" / f"slide_{SLIDE}" / "lfc_agg.json"

agg = json.loads(agg_path.read_text())["L2"]
print("read", agg_path, "\n")

rows = [{"perturbation": g,
         "lfc_spearman": v["spearman_mean"],
         "lfc_rmse": v["lfc_dist_mean"],
         "n_niches": v["n_niches"],
         "n_scored_genes": v["n_scored_genes"]}
        for g, v in agg.items()]
per_pert = pd.DataFrame(rows).sort_values("lfc_spearman", ascending=False)
print(f"slide {SLIDE}: {len(per_pert)} perturbations, "
      f"mean LFC Spearman {per_pert.lfc_spearman.mean():.4f}, "
      f"mean LFC RMSE {per_pert.lfc_rmse.mean():.4f}")
per_pert

## 8. The five slides together - the within-niche number

The reported figure is the mean over every **record**, a record being one
(perturbation, slide, niche) triple. The cell below reads the five per-slide result files and
aggregates them.

By default it reads the **shipped** results, so it reproduces the published number exactly. Point
`RESULT_ROOT` at `args.outdir.parent` instead once you have run step 7 for all five slides
(`for s in SLIDES: bench.eval_seen_slide(spac_args("seen", s, WORKDIR/"seen_eval"), s)`).

In [ ]:
RESULT_ROOT = SHIPPED["seen"]        # or: WORKDIR / "seen_eval"

records = []
for s in SLIDES:
    p = RESULT_ROOT / "patch" / f"slide_{s}" / "lfc_agg.json"
    if not p.exists():
        print(f"  slide {s}: no result at {p}")
        continue
    for gene, v in json.loads(p.read_text())["L2"].items():
        records.append({"slide": s, "perturbation": gene,
                        "lfc_spearman": v["spearman_mean"],
                        "lfc_rmse": v["lfc_dist_mean"],
                        "n_niches": v["n_niches"]})
seen_df = pd.DataFrame(records)

print(f"{len(seen_df)} perturbation x slide records over {seen_df.slide.nunique()} slides")
print(f"\nWITHIN-NICHE (code name: seen)")
print(f"  LFC Spearman : {seen_df.lfc_spearman.mean():.4f}  "
      f"(s.e.m. {seen_df.lfc_spearman.std(ddof=1) / np.sqrt(len(seen_df)):.4f})")
print(f"  LFC RMSE     : {seen_df.lfc_rmse.mean():.4f}  "
      f"(s.e.m. {seen_df.lfc_rmse.std(ddof=1) / np.sqrt(len(seen_df)):.4f})")

seen_df.groupby("slide").agg(n_records=("perturbation", "size"),
                             lfc_spearman=("lfc_spearman", "mean"),
                             lfc_rmse=("lfc_rmse", "mean")).round(4)

## 9. The cross-slide setting

In the cross-slide setting a perturbation is removed from **every** slide, so the model has never
seen it in training and has to infer its effect from related perturbations. The evaluation is
therefore organised differently: instead of niches of a held-out patch, it walks a panel of
(gene, target slide) **units** listed in `benchmark_v3/eval_panel.json`. Each unit names the
donor slides where the gene was still visible.

`eval_unseen_all` walks the whole panel. It is faster than the within-niche pass - one gene per
unit rather than the full panel - but still tens of minutes. The cell shows the panel, runs
**one** unit as a demonstration, and then reads the shipped results for the reported number.

In [ ]:
panel = json.loads((PREPARED / "benchmark_v3" / "eval_panel.json").read_text())
units = pd.DataFrame([{"gene": u["gene"], "target_slide": u["target_slide"],
                       "donor_slides": ",".join(map(str, u.get("donor_slides", [])))}
                      for u in panel["eval_units"]])
print(f"{len(units)} cross-slide evaluation units")
units

In [ ]:
%%time
RUN_ONE_UNSEEN_UNIT = True    # runs a single unit end-to-end from the unseen checkpoint

if RUN_ONE_UNSEEN_UNIT:
    u_args = spac_args("unseen", outdir=WORKDIR / "unseen_eval_demo")
    one = dict(panel)
    one["eval_units"] = panel["eval_units"][:1]         # just the first unit
    demo_panel_dir = WORKDIR / "demo_panel" / "benchmark_v3"
    demo_panel_dir.mkdir(parents=True, exist_ok=True)
    (demo_panel_dir / "eval_panel.json").write_text(json.dumps(one))
    # the holdout coordinates live beside the real panel; symlink the rest of the directory
    for f in (PREPARED / "benchmark_v3").glob("*"):
        tgt = demo_panel_dir / f.name
        if not tgt.exists():
            tgt.symlink_to(f)
    u_args.benchmark_v3_dir = demo_panel_dir
    bench.eval_unseen_all(u_args)
    print("\ndemo unit written to", u_args.outdir)

In [ ]:
unseen_records = []
for d in sorted((SHIPPED["unseen"] / "unseen_pert").glob("*__slide_*")):
    p = d / "lfc_scores.json"
    if not p.exists():
        continue
    gene, slide = d.name.split("__slide_")
    # lfc_scores.json is {"L2": {gene: [record]}} -- L2 is the unassigned source level,
    # and a cross-slide unit has exactly one record (niche = -1, the whole slide).
    rec = json.loads(p.read_text())["L2"][gene][0]
    unseen_records.append({"gene": gene, "slide": slide,
                           "n_src_cells": rec.get("n_src"),
                           "n_measured_ko": rec.get("n_real_g"),
                           "lfc_spearman": rec.get("spearman"),
                           "lfc_rmse": rec.get("lfc_dist")})
unseen_df = pd.DataFrame(unseen_records).dropna(subset=["lfc_spearman"])

print("CROSS-SLIDE (code name: unseen)")
print(f"  {len(unseen_df)} units")
print(f"  LFC Spearman : {unseen_df.lfc_spearman.mean():.4f}")
print(f"  LFC RMSE     : {unseen_df.lfc_rmse.mean():.4f}")
unseen_df.sort_values("lfc_spearman", ascending=False)

## 10. CONCERT next to the published methods

Every method is rescored under **one** rule into a per-record CSV, so the comparison does not
depend on each method's own reporting conventions. A record is one (perturbation, slide, niche)
triple.

**The comparison is paired, one competitor at a time.** The methods cover very different subsets
of the 129 within-niche records - Biolord, scGEN and Steamboat cover all of them, GEARS 80,
CelComen 15, CellAgentChat 11 - so intersecting *all* of them at once leaves nothing to compare.
What the reported comparison does instead is take, for each competitor separately, the records
that competitor and CONCERT both cover, and compare the two on exactly those. The `n_pairs`
column of the shipped statistics table is that count, and the cell below reproduces it.

Comparing unpaired means instead would be the trap this avoids: a method that skipped the hard
records would look better simply for having skipped them.

CONCERT appears under the method key `concert_v3_seen` / `concert_v3_unseen`; every method's
label column reads its display name.

In [ ]:
def paired_comparison(csv_path, concert_key):
    df = pd.read_csv(csv_path)
    df["record"] = df.gene.astype(str) + "|" + df.slide.astype(str) + "|" + df.niche_key.astype(str)
    con = df[df.method == concert_key].set_index("record")
    rows = []
    for m, sub in df[df.method != concert_key].groupby("method"):
        sub = sub.set_index("record")
        shared = con.index.intersection(sub.index)
        if len(shared) == 0:
            continue
        rows.append({
            "competitor": sub.method_label.iloc[0],
            "n_pairs": len(shared),
            "concert_rho": con.loc[shared, "lfc_spearman"].mean(),
            "competitor_rho": sub.loc[shared, "lfc_spearman"].mean(),
            "concert_rmse": con.loc[shared, "lfc_dist"].mean(),
            "competitor_rmse": sub.loc[shared, "lfc_dist"].mean(),
        })
    out = pd.DataFrame(rows)
    out["rho_advantage"] = out.concert_rho - out.competitor_rho
    out["rmse_advantage"] = out.competitor_rmse - out.concert_rmse   # positive = CONCERT lower
    return out.sort_values("n_pairs", ascending=False), len(con)

seen_cmp, n_concert = paired_comparison(
    PAPER_NUMBERS / "spac_seen_lfc_dist_per_record.csv", "concert_v3_seen")
print(f"WITHIN-NICHE: CONCERT covers {n_concert} records; paired against each competitor:")
display(seen_cmp.round(4))

unseen_cmp, n_concert_u = paired_comparison(
    PAPER_NUMBERS / "spac_unseen_lfc_dist_per_record.csv", "concert_v3_unseen")
print(f"\nCROSS-SLIDE: CONCERT covers {n_concert_u} records; paired against each competitor:")
display(unseen_cmp.round(4))

The shipped statistics table carries the same `n_pairs` per competitor. Matching it is the check
that the pairing above is the one the paper used.

In [ ]:
shipped_stats = pd.read_csv(PAPER_NUMBERS / "spacseq_seen_concert_vs_competitor_lfc_rho_stats_summary.csv")
check = (seen_cmp[["competitor", "n_pairs"]]
         .merge(shipped_stats[["method", "n_pairs"]].rename(
             columns={"method": "competitor", "n_pairs": "n_pairs_shipped"}),
             on="competitor", how="left"))
check["matches"] = check.n_pairs == check.n_pairs_shipped
display(check)
print("all counts match:", bool(check.matches.all()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (cmp_df, name) in zip(axes, [(seen_cmp, "within-niche"), (unseen_cmp, "cross-slide")]):
    d = cmp_df.sort_values("concert_rho")
    y = np.arange(len(d))
    ax.barh(y - 0.2, d.concert_rho, 0.4, color="#b2182b", label="CONCERT")
    ax.barh(y + 0.2, d.competitor_rho, 0.4, color="#9ab8d4", label="competitor")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{c}  (n={n})" for c, n in zip(d.competitor, d.n_pairs)])
    ax.axvline(0, lw=0.8, color="0.4")
    ax.set_xlabel(f"LFC Spearman on paired records ({name})", fontsize=13)
    ax.tick_params(labelsize=10)
    ax.legend(fontsize=10)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
fig.tight_layout()
plt.show()

## What to watch out for

* **The lengthscale band is not in the weights.** `--fpert_ls_min` / `--fpert_ls_max` must be
  passed at evaluation exactly as at training. Leave them out and the same weights render a
  different spatial field - the run completes and produces plausible, wrong numbers.
* **The vocabulary is not rebuilt.** Evaluation reads `pert_map.json` from beside the checkpoint.
  A checkpoint without it cannot be scored; do not regenerate it from the data.
* **Compare against one competitor at a time, on the records they share.** The methods cover
  very different subsets, and two of them cover barely a tenth of the records - an
  all-method intersection is empty here. See step 10. (The Perturb-map benchmark is the
  other way round: there the methods overlap enough that the intersection is the rule.)
* **`seen` / `unseen` are directory names, not manuscript terms.** They are *within-niche
  prediction* and *cross-slide prediction* in text.
* **E-distance is retired.** It still appears in some of the JSON files. The reported metrics are
  LFC RMSE and LFC Spearman.
* **Storage.** `/n/holylfs06` is close to its group quota, and writes there can fail *and
  truncate the target to zero*. This notebook writes to `/n/holylabs` and only reads from
  holylfs06.

## Where things are

| | |
|---|---|
| model source | `codebase/models/concert_benchmark/` |
| scoring code | `codebase/benchmark_evaluation/` |
| experiment scripts | `codebase/experiments/01_spac_seq_benchmark/` |
| reported tables | `paper_package/numbers/benchmark_spac_seq/` |